> ### ⚠️ Nota metodológica — abordagem superada
>
> Este notebook usa `ImageFolder` + `random_split`, que divide os dados **por
> imagem**. Como cada paciente contribui com centenas de células da mesma
> lâmina, essa divisão espalha células do mesmo paciente entre treino e teste, e
> o modelo passa a acertar em parte por reconhecer o paciente — não a doença. É
> vazamento de dados (*data leakage*).
>
> O pipeline definitivo do trabalho está em `src/dados/dataset.py`, que divide
> **por paciente**. O notebook é mantido aqui como registro da evolução do
> projeto e como base do experimento que mede o efeito do vazamento: treinando
> as duas divisões com hiperparâmetros idênticos, a divisão aleatória infla o
> recall em +0,0216 e a acurácia em +0,0150 (ver `outputs/comparacao_divisoes.md`).
>
> **Não use esta divisão para reportar resultados.**

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt
import numpy as np

# 1. Definindo o tamanho do lote e o caminho dos dados
caminho_dataset = "../data/processed/dataset_binario"
tamanho_lote = 32 # Lotes de 32 imagens garantem estabilidade na VRAM da placa de vídeo

# 2. Criando o pipeline de transformação (Visão Computacional)
# Toda imagem que passar por aqui será redimensionada e transformada em Matriz Matemática (Tensor)
transformacoes = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # Padrão exigido por modelos profissionais
])

# 3. Lendo a pasta raiz
dataset_completo = datasets.ImageFolder(root=caminho_dataset, transform=transformacoes)
print(f"Total de imagens mapeadas com sucesso pelo PyTorch: {len(dataset_completo)}")
print(f"Classes identificadas (Gabarito): {dataset_completo.class_to_idx}\n")

# 4. Divisão Matemática: 70% Treino, 15% Validação, 15% Teste
tamanho_total = len(dataset_completo)
tamanho_treino = int(0.7 * tamanho_total)
tamanho_val = int(0.15 * tamanho_total)
tamanho_teste = tamanho_total - tamanho_treino - tamanho_val

treino_data, val_data, teste_data = random_split(
    dataset_completo, 
    [tamanho_treino, tamanho_val, tamanho_teste],
    generator=torch.Generator().manual_seed(42) # Seed fixa para garantir que o seu TCC seja 100% reproduzível
)

# 5. Criando os Dataloaders (Os motores que vão alimentar a GPU)
train_loader = DataLoader(treino_data, batch_size=tamanho_lote, shuffle=True)
val_loader = DataLoader(val_data, batch_size=tamanho_lote, shuffle=False)
test_loader = DataLoader(teste_data, batch_size=tamanho_lote, shuffle=False)

print(f"Arquitetura de Lotes (Dataloaders) pronta:")
print(f"- Lotes de Treino gerados: {len(train_loader)} (com {tamanho_lote} imagens cada)")
print(f"- Lotes de Validação gerados: {len(val_loader)}")
print(f"- Lotes de Teste gerados: {len(test_loader)}")

# 6. Teste de Fumaça: Pegando o PRIMEIRO lote para ver se o sistema não engasga
imagens_batch, labels_batch = next(iter(train_loader))
print(f"\n✅ Teste de Fumaça passou! Formato matemático do lote que vai para a GPU: {imagens_batch.shape}")